In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from vncorenlp import VnCoreNLP
import pandas as pd
import re
import os

llm = ChatOllama(model="myaniu/qwen2.5-1m")  # hoặc "llama3:8b-instruct"

def summary_abstractive(text, grade, length):
    prompt = [
        SystemMessage(content=f"Bạn là giáo viên dạy môn học tiếng Việt cho học sinh tiểu học lớp {grade} tại Việt Nam."),
        HumanMessage(content=f"""
        Đầu tiên tôi muốn bạn biết tóm tắt diễn giải là gì ?. Tóm tắt diễn giải (Abstractive Summarization): là phương pháp tóm tắt trong đó hệ thống hiểu toàn bộ nội dung tổng thể và tái diễn đạt lại các ý chính ngắn gọn và tự nhiên trong một văn bản mới, tương tự như cách con người tóm tắt. Hệ thống sẽ tạo ra các câu văn ngắn gọn, mạch lạc, tự nhiên và dễ hiểu.
        Tiếp theo tôi muốn bạn đọc đoạn văn sau và thực hiện tóm tắt diễn giải văn bản sao cho phù hợp với học sinh lớp {grade}.
        Với các yêu cầu như sau:
        - Tóm tắt diễn giải phải đảm bảo tính chính xác, tính logic, tính hợp lý, không mất ý nghĩa của văn bản gốc.
        - HÃY TUÂN THỦ VÀ THỰC HIỆN TÓM TẮT VĂN BẢN TRONG {length} CÂU. Phải chú trọng đến độ liên kết giữa các câu, giữa các đoạn văn. Nội dung mạch lạc, trôi chảy, liên kết. Bản tóm tắt phải dễ hiểu, lời lẽ tự nhiên, từ vựng phù hợp, đơn giản với học sinh lớp {grade}.
        - Cuối cùng hãy chuẩn hóa bản tóm tắt thành dạng văn bản hoàn chỉnh. Viết trên cùng 1 dòng.
        - KHÔNG THÊM CÁC DÁNH DẤU, CHỈ MỤC, CHÚ THÍCH, GHI CHÚ, ...
        - KẾT QUẢ CHỈ GỒM VĂN BẢN ĐƯỢC TÓM TẮT. KHÔNG THÊM TẤT CẢ CÁC YẾU TỐ, THÔNG TIN NGOÀI LỀ KHÁC.
        ĐOẠN VĂN:
        {text}

        KẾT QUẢ:
        """)
    ]
    response = llm(prompt)
    return response.content.strip()

vncorenlp = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg"
)

def tokenize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    sentences = vncorenlp.tokenize(text)
    tokens = []
    for sent in sentences:
        tokens.extend(sent)
    return tokens

def load_grade_vocab(grade, vocab_dir="../../../datasets/grade_vocab"):
    vocab = set()
    for g in range(1, grade + 1):
        path = os.path.join(vocab_dir, f"grade_{g}.txt")
        with open(path, encoding="utf-8") as f:
            for line in f:
                vocab.add(line.strip().lower())
    return vocab

def vocab_coverage(text, vocab):
    tokens = tokenize_text(text)
    if len(tokens) == 0:
        return 0.0
    valid = sum(1 for t in tokens if t in vocab)
    return valid / len(tokens)


data = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data_DG_with_summary5.xlsx")
df = data.copy()
max_retry = 3
threshold = 0.85 #giai thich pp nao
annotator = VnCoreNLP(
    "../../VnCoreNLP-master/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx2g'
)

for idx, row in df.iterrows():
    if pd.isna(row["summary"]) or row["summary"] == "":
        content = str(row["content"])
        grade = int(row["grade"])
        sentences = annotator.tokenize(content)
        num_sentences = len(sentences)
        length = num_sentences // 3
        vocab = load_grade_vocab(grade)
        dict_result = {}
        for i in range(max_retry):
            summary_result = summary_abstractive(content, grade, length)
            score = vocab_coverage(summary_result, vocab)
            dict_result[i] = {
                "summary": summary_result,
                "score": score
            }
            if score >= threshold:
                break
        best_summary = max(dict_result.values(), key=lambda x: x["score"])["summary"] if dict_result else ""
        df.loc[idx, "summary"] = best_summary

df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\Data_DG_with_summary6.xlsx", index=False)

#xem lai sau nay